# 02 - 音訊前處理（驗證）

批次前處理由 `src/run_preprocessing.py` 執行：
```bash
cd src && python run_preprocessing.py
```

處理內容：
- 篩選 6 類情緒（移除 calm, surprise）
- Resample 16kHz + mono + peak normalize
- 不做 pad/truncate，保留完整長度

本 notebook 用於**驗證前處理結果**。

In [ ]:
import pandas as pd
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "audio_16k"

df = pd.read_csv(PROJECT_ROOT / "data" / "metadata.csv")
print(f"樣本數：{len(df):,}")
print(f"情緒類別：{sorted(df['emotion'].unique())}")
print(f"欄位：{df.columns.tolist()}")
print(f"\n各資料集筆數：")
print(df["dataset"].value_counts().sort_index().to_string())

## 1. 原始取樣率確認

各資料集的原始取樣率不同，前處理統一為 16kHz。

In [ ]:
# 抽樣每個資料集 3 筆，檢查原始取樣率
print("各資料集原始取樣率抽樣：\n")
for dataset in sorted(df["dataset"].unique()):
    subset = df[df["dataset"] == dataset].sample(3, random_state=42)
    for _, row in subset.iterrows():
        full_path = PROJECT_ROOT / row["filepath"]
        info = sf.info(str(full_path))
        print(f"  [{dataset:10s}] sr={info.samplerate:>6} Hz, "
              f"ch={info.channels}, "
              f"dur={info.duration:.2f}s")
    print()

## 2. 前處理結果驗證

確認前處理後的音訊：取樣率=16kHz、mono、peak=1.0。

In [ ]:
# 確認 processed_path 欄位存在
assert "processed_path" in df.columns, "請先執行 src/run_preprocessing.py"

print("前處理後音訊抽樣驗證：\n")
sample = df.sample(10, random_state=42)
for _, row in sample.iterrows():
    full_path = PROJECT_ROOT / row["processed_path"]
    info = sf.info(str(full_path))
    audio, sr = librosa.load(str(full_path), sr=None)
    peak = np.max(np.abs(audio))
    print(f"  [{row['dataset']:10s}] sr={info.samplerate:>6} Hz, "
          f"ch={info.channels}, "
          f"dur={info.duration:.2f}s, "
          f"peak={peak:.4f}, "
          f"emotion={row['emotion']}")

# 統計
total_size = sum(f.stat().st_size for f in OUTPUT_DIR.glob("*.wav"))
file_count = len(list(OUTPUT_DIR.glob("*.wav")))
print(f"\n目錄統計：{file_count:,} 檔案, {total_size / 1e9:.2f} GB")